* Connect to weather API, get output into a table
* One table per university with all available data then combine to comprehensive u-weather table
    * Start easy then get more challenging
* One table per university with u-data (e.g., # of students), combine to comprehensive u-data table
    * Analyze data, define "severe" weather circumstances, apply to next summary table
* Combine summary statistics from u-data and u-weather to final table to answer hmwk questions.

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import json
import re

In [ ]:
# Create dictionary of universities and lat/long
places = {
    'u_miss': {'latitude': 34.365, 'longitude': -89.5383, 'state': 'MS', 'full_name': 'University of Mississippi'},
    'tulane': {'latitude': 29.9401, 'longitude': -90.1207, 'state': 'LA', 'full_name': 'Tulane University'},
    'g_tech': {'latitude': 33.7756, 'longitude': -84.3963, 'state': 'GA', 'full_name': 'Georgia Tech'},
    'u_ark': {'latitude': 36.0692, 'longitude': -94.1756, 'state': 'AR','full_name': 'University of Arkansas'},
    'u_bama': {'latitude': 33.2113, 'longitude': -87.5398, 'state': 'AL', 'full_name': 'University of Alabama'}
}

# Access lat/long values within Places dictionary
lats_longs = places.values()
# print(lats_longs)

# Bring all lats/longs into a comma-separated list of strings
all_lats = ','.join(str(d['latitude']) for d in lats_longs)
all_longs = ','.join(str(d['longitude']) for d in lats_longs)

# Add base url for API
base_url = "https://archive-api.open-meteo.com/v1/archive"

# Add parameters into a dictionary
api_params = {
    'latitude': all_lats,
    'longitude': all_longs,
    'start_date': '2026-01-01',
    'end_date': '2026-01-31',
    'daily': ['precipitation_sum', 'snowfall_sum', 'precipitation_hours', 'temperature_2m_max', 'temperature_2m_mean', 'temperature_2m_min'],
    'timezone': 'auto',
    'temperature_unit': 'fahrenheit',
    'wind_speed_unit': 'mph',
    'precipitation_unit': 'inch'
}

# Get response
response = requests.get(base_url, params=api_params)
data = response.json()
## Preview the data
# data

In [ ]:
# Use pandas to bring data into a data frame
weather_df = pd.DataFrame(data)

# Create a list of university names
school_names = list(places.keys())

# Add series of names to weather_df
weather_df['school'] = school_names

# Check the results
weather_df

In [ ]:
# Instantiate list of dataframes
all_frames = []

for index, row in weather_df.iterrows():
    # Get contents of daily dict into a temp dataframe
    tempdf = pd.DataFrame(row['daily'])
    # Add school name to df
    tempdf['school'] = row['school']
    # Append to final dataframe
    all_frames.append(tempdf)

# Concatenate list of 5 frames into one big frame
final_df = pd.concat(all_frames, ignore_index=True)

# # Review data
print(final_df)

In [ ]:
# Update headers with units for later use
# Grab units from weather df
units = weather_df.iloc[0]['daily_units']

# Create renaming dictionary -- Loop through the keys and values (e.g., precip_sum and inch)
# and create a map: {'precipitation_sum': 'precipitation_sum_inch'}
new_names = {col: f"{col}_{unit}" for col, unit in units.items() if col in final_df.columns}

# Rename dataframes with more detail
final_df = final_df.rename(columns=new_names)

# Check final column output
print(final_df.columns)

In [ ]:
def fetch_enrollment(place_key, url, search_word, regex_pattern, places_dict):
    """
    Scrapes a specific URL for enrollment numbers and updates the places dictionary.
    """
    
    # Standard headers to look like a browser (AI tip: prevents 403 Forbidden errors)
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

    try:
        response = requests.get(url, headers=headers)

        # If initial call doesn't work, hop out of the function
        if response.status_code != 200:
            print(f"[{place_key}] Failed: Status Code {response.status_code}")
            return

        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Finds the text node string based on the search word
        text_node = soup.find(string=re.compile(search_word))
        
        if not text_node:
            print(f"[{place_key}] Keyword '{search_word}' not found on page.")
            return

        # UPDATE: Instead of just grabbing the immediate .parent, look up the HTML tree.
        # Look for a "Block Level" container (paragraph, header, div, or list item).
        container = text_node.find_parent(['p', 'div', 'h2', 'h3', 'li'])
        
        if container:
            # Get all text inside that container, including the number
            target_content = container.get_text()
            
            # Remove newlines so the regex can match across line breaks
            target_content = target_content.replace('\n', ' ')
            
            # Run regex on the full container text
            match = re.search(regex_pattern, target_content)
            
            if match:
                # Clean the number (remove commas)
                raw_number = match.group(1)
                enrollment_count = int(raw_number.replace(',', ''))
                
                # Update the dictionary
                places_dict[place_key]['total_pop'] = enrollment_count
                print(f"[{place_key}] Success! Enrollment: {enrollment_count}")
            else:
                # Only prints if we found the section but the regex pattern didn't fit
                print(f"[{place_key}] Found container, but regex failed. Content snippet: '{target_content[:50]}...'")
        else:
            # Only prints if the text was found floating outside a valid tag (rare)
            print(f"[{place_key}] Found text, but couldn't find a valid parent container.")

    except Exception as e:
        print(f"[{place_key}] Error: {e}")

In [ ]:
# Use dictionary to compile sources to scrape so I can iterate over them with the function
scraping_config = {
    # University of Mississippi
    'u_miss': {
        'url': 'https://olemiss.edu/news/2025/11/um-reaches-record-enrollment-for-third-straight-year/index.html',
        'search_word': 'welcomed', 
        'regex_pattern': r'welcomed\s+([\d,]+)\s+students'
    },

    # Trouble with the iframe--can't do simple HTML extraction
    # Tulane University
    'tulane': {
        'url': 'https://en.wikipedia.org/wiki/Tulane_University', 
        'search_word': 'Undergraduates',
        'regex_pattern': r'Students\D*([\d,]{4,})'
    },
    
    # Georgia Tech
    'g_tech': {
        'url': 'https://en.wikipedia.org/wiki/Georgia_Institute_of_Technology',
        'search_word': 'Students',
        'regex_pattern': r'Students\D*([\d,]{4,})'
    },
    
    # University of Arkansas
    'u_ark': {
        'url': 'https://www.uark.edu/about/quick-facts.php',
        'search_word': 'Record student enrollment',
        'regex_pattern': r'([\d,]+)\s+Record student enrollment'
    },
    
    # University of Alabama
    'u_bama': {
        'url': 'https://www.ua.edu/about/quick-facts/',
        'search_word': 'welcoming',
        'regex_pattern': r'welcoming\s+([\d,]+)\s+students'
    }
}

In [ ]:
# Loop through scraping dictionary to get each school and its attributes, 

print("Starting Scraping Job...")
print("-" * 30)

for school_key, config in scraping_config.items():

    # Used TBD as placeholder for schools as they were added, kept.
    if config['url'] != 'TBD':
        print(f"Scraping {school_key}...")
        
        fetch_enrollment(
            place_key=school_key,
            url=config['url'],
            search_word=config['search_word'], 
            regex_pattern=config['regex_pattern'],
            places_dict=places  # Pass your master dictionary to be updated
        )
    else:
        print(f"Skipping {school_key} (No URL configured)")

print("-" * 30)
print("Job Complete. Updated Data:")
print(places)

In [ ]:
# Update final_df with student populations, states, and full university names for final output
pop_lookup = {school: data['total_pop'] for school, data in places.items()}
state_lookup = {school: data['state'] for school, data in places.items()}
name_lookup = {school: data['full_name'] for school, data in places.items()}

# Map values to final_df before analysis
final_df['student_pop'] = final_df['school'].map(pop_lookup)
final_df['state'] = final_df['school'].map(state_lookup)
final_df['university_name'] = final_df['school'].map(name_lookup)

In [ ]:
def analyze_southern_winter(input_df):
    # Running analysis on copy to be sure nothing messes up our original data
    df = input_df.copy()

    # Assign weather data to variables for analysis
    snow_col = 'snowfall_sum_inch'
    precip_col = 'precipitation_sum_inch'
    temp_min_col = 'temperature_2m_min_°F'

    # Define rules for severe winter weather--thresholds lower for southern states
    # Based on severe weather event criteria for locations on NOAA's website

    # Get Counts for each type of severe weather day
    # Rule 1: Severe Snow is >= 2.0 inches
    df['is_snow_day'] = df[snow_col].fillna(0) >= 2.0

    # Rule 2: Severe Ice is Rain >= 0.10" AND Freezing
    df['is_ice_day'] = (df[precip_col].fillna(0) >= 0.10) & (df[temp_min_col] < 32.0)

    # Rule 3: Severe Cold is <= 15 F
    df['is_cold_day'] = df[temp_min_col] <= 15.0

    # Rule 4: Count impact based on any of the above
    df['is_severe_impact'] = df['is_snow_day'] | df['is_ice_day'] | df['is_cold_day']

    # Get dates of severe weather per university
    df['severe_date'] = df.apply(lambda x: x['time_iso8601'] if x['is_severe_impact'] else None, axis=1)
    
    # Calculating measurement of precipitation on days where precipitation was likely ice
    ## This was more complicated than a group/agg like some others
    df['ice_storm_precip'] = df.apply(lambda x: x[precip_col] if x['is_ice_day'] else 0, axis=1)

    # Group and aggregate report results
    summary = df.groupby('school').agg({
        'university_name': 'first',         # including full name
        'state': 'first',                   # state
        'student_pop': 'max',               # student population
        'is_severe_impact': 'sum',          # and count of days impacted

        # Dates need collected into a list so there's still just one record per university
        'severe_date': lambda x: [d for d in x if d is not None],

        # Add detailed weather columns to back up assertion
        'is_snow_day': 'sum',
        snow_col: 'max',
        'is_ice_day': 'sum',
        'ice_storm_precip': 'max',      
        'is_cold_day': 'sum',
        temp_min_col: 'min'
    })

    # Rename columns before final output
    summary = summary.rename(columns={
            'university_name': 'University Name',
            'state': 'State',
            'student_pop': 'Total Students',
            'is_severe_impact': 'Total Severe Days',
            'severe_date': 'List of Severe Dates', 
            'impacted_student_days': 'Impacted Student-Days',
            'students_affected': 'Students Affected',
            'is_snow_day': 'Snow Days',
            snow_col: 'Max Snow (Inches)',
            'is_ice_day': 'Ice Days',
            'ice_storm_precip': 'Max Ice (Inches)',
            'is_cold_day': 'Severe Cold Days',
            temp_min_col: 'Min Temp'
        })

# Students Affected (only counting if there was at least one severe day)
    summary['Students Affected'] = summary['Total Students']
    summary.loc[summary['Total Severe Days'] == 0, 'Students Affected'] = 0
    
    # 2. Impacted Student-Days (Students * Days)
    summary['Impacted Student-Days'] = summary['Total Students'] * summary['Total Severe Days']
    
    # Reorder columns to put the most important stuff first
    final_order = [
        'University Name', 'State', 'Total Students', 'Students Affected', 'Impacted Student-Days',
        'Total Severe Days', 'List of Severe Dates',
        'Snow Days', 'Max Snow (Inches)', 'Ice Days', 'Max Ice (Inches)', 'Min Temp'
    ]
    
    return summary[final_order].sort_values('Impacted Student-Days', ascending=False)

# Run it
final_report = analyze_southern_winter(final_df)
final_report

## Data Engineering - Assignment 04
### Universities Impacted by Severe Winter Weather (January 2026)

#### Process Review and Level-Setting
I started this assignment by picking a general category for universities to research. I opted for five of the states south of Missouri: **Arkansas, Alabama, Mississippi, Georgia, and Louisiana.** As with previous assignments, I bulleted my steps at the onset so that I could think through the process as I was gathering and reviewing data. I could tell immediately that the differences between the websites would be a challenge, so I focused on getting a steady connection (and ingestion) of data from **Open-Meteo** for weather data. 

After that was accomplished, I went through and looked at the Google Chrome "inspect" page for each of my websites. Some elements were very easy to find, not buried in deep structures, while others (like Tulane) used an **iframe** on their website. I was only vaguely familiar with what these were, but realized from reviewing the "inspect" page that this was referencing a PowerBI data source, and that I would not be able to use it for the HTML assignment. Fortunately, I had already requested access to Wikipedia for this assignment, and Tulane's Wiki page had the information I needed. 

I realized quickly that the layering and different structures would be a significant challenge. I went from citing a paragraph (`<p>`) to a list element, to simply relying on a search-word with regular expressions. I used this iterative search approach to find key words that helped me zero-in on the enrollment data, regardless of its location in the page. 

---

### Code Review
The structure of my code isn't exactly where I wanted it, but I'm overall proud. Not all of it was written directly from my brain, but I thought through the process, order, and efficiency with Gemini. 

* **Configuration:** I set up the configuration at the head of the code with a `places` dictionary that allowed fundamental setting of parameters for the weather extraction and university identification.
* **Ingestion:** I made my HTML call to the website and pulled back a high-level summary which had some nested dictionaries and lists for each day in January per school. I formatted the nested data as data frames grouped by their respective schools and concatenated them into a "final" data frame with monthly weather information. 
* **The `fetch_enrollment` Function:** The prized piece of this code is probably the `fetch_enrollment` function that works in combination with a scraping configuration dictionary and `for` loop to zero-in on enrollment information to add it back to the dictionary. 

Final touches on the `final_df` dataframe with my January weather data are adding the student population, state, and full university name from my, now enhanced, `places` dictionary.

---

### Analysis, Results, and Reflection
Analysis for the assignment is done with a function that uses snowfall, precipitation, ice accumulation, and temperature thresholds (**bespoke to southern severe weather standards based on NOAA website information**) to aggregate and display grouped weather data per university. 

This logic calculates the required **"Impacted Student-Days"** as well as the **Severe Dates**. I would have loved more time for this assignment, and I feel like I already sank in at least 8 hours, but I'm beginning to understand where the experience in all of this truly comes in handy.